In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import os

class GPTChatClient:
    """Loads GPT config from .env, wraps chat(), and prettifies outputs."""
    def __init__(
        self,
        system_prompt: str | None = None,
        temperature: float = 0,
        render_markdown: bool = True,
        title: str | None = "GPT-5.2 Response",
    ):
        load_dotenv()
        api_key = os.getenv("OPENROUTER_API_KEY")
        if not api_key:
            raise RuntimeError("请先在 .env 中设置 OPENROUTER_API_KEY")
        base_url = os.getenv("OPENROUTER_API_BASE", "https://openrouter.ai/api/v1")
        model = os.getenv("OPENROUTER_MODEL", "openai/gpt-5.2:online")
        self._client = OpenAI(api_key=api_key, base_url=base_url)
        self._model = model
        self._system_prompt = system_prompt
        self._default_temperature = temperature
        self._render_markdown = render_markdown
        self._markdown_title = title

    def chat(self, prompt: str | list[dict], temperature: float | None = None) -> str:
        """Send either a string prompt or message list, render Markdown, and return text."""
        messages: list[dict] = []
        if self._system_prompt:
            messages.append({"role": "system", "content": self._system_prompt})
        if isinstance(prompt, str):
            messages.append({"role": "user", "content": prompt})
        else:
            messages.extend(prompt)

        resp = self._client.chat.completions.create(
            model=self._model,
            messages=messages,
            temperature=self._default_temperature if temperature is None else temperature,
        )
        content = resp.choices[0].message.content or ""
        if self._render_markdown:
            self._display_markdown(content)
        else:
            print(content)
        return content

    def _display_markdown(self, content: str) -> None:
        body = (content.strip() or "_No content returned._").replace("\n", "\n\n")
        title = f"### {self._markdown_title}\n\n" if self._markdown_title else ""
        display(Markdown(f"{title}{body}"))

In [2]:
system_prompt = '''
You are EnzymeAnnotator. Your ONLY tool is web browsing. The user may give a UniProt accession and ask specific questions. Do: fetch → extract → answer ONLY what they asked, with citations. Never invent.

Steps
1) Open the exact UniProt entry for the accession (search: "<ACC> site:uniprot.org") and confirm it matches.
2) Pull only the fields needed to answer the user’s request (enzyme? EC? active-site residues? etc.).
3) If UniProt lacks the requested detail, optionally check RCSB PDB / InterPro / M-CSA(=CSA) to support that specific item.

Rules
- Every factual claim needs a URL citation (prefer UniProt).
- Mark curated/experimental vs predicted/inferred; if missing, say “not found in authoritative sources”.
- If the accession is invalid/ambiguous, say so and ask for the minimum clarification.

'''

In [3]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CCCCC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCC(=O)OC(COC(=O)CCCC/C=C\C/C=C\C/C=C\CCCCC)COP(=O)(O)OCCNC.C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1O>>CCCCC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCC(=O)OC(COC(=O)CCCC/C=C\C/C=C\C/C=C\CCCCC)COP(=O)(O)OCCN(C)C ?")

### GPT-5.2 Response

Your reaction is a classic SAM-dependent **headgroup N-methylation** on a phospholipid: the substrate has a **phosphatidyl-N-methylethanolamine** headgroup (–NHCH₃), and the product is the **phosphatidyl-N,N-dimethylethanolamine** headgroup (–N(CH₃)₂). The other reagent is **S-adenosyl-L-methionine (SAM)**, and the byproduct is **S-adenosyl-L-homocysteine (SAH)**.

Recommended enzyme (EC number):
• **EC 2.1.1.71 — phosphatidyl-N-methylethanolamine N-methyltransferase** (also called phosphatidylmonomethylethanolamine methyltransferase). This enzyme catalyzes exactly:
SAM + phosphatidyl-N-methylethanolamine ⇌ SAH + phosphatidyl-N,N-dimethylethanolamine. ([Expasy ENZYME][1])

Extra note (useful for pathway design) ✅
If you start one step earlier from **phosphatidylethanolamine (–NH₂)**, the first methylation step is typically associated with:
• **EC 2.1.1.17 — phosphatidylethanolamine N-methyltransferase** (PE → phosphatidyl-N-methylethanolamine). ([维基百科][2])

[1]: https://enzyme.expasy.org/EC/2.1.1.71?utm_source=chatgpt.com "ENZYME - 2.1.1.71 phosphatidyl-N-methylethanolamine N- ..."
[2]: https://en.wikipedia.org/wiki/Phosphatidylethanolamine_N-methyltransferase?utm_source=chatgpt.com "Phosphatidylethanolamine N-methyltransferase"


In [4]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)([O-])OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](OP(=O)([O-])[O-])[C@@H]3O)[C@@H](O)[C@H]2O)c1.[H+].[O-]c1cc2c([O-])cc(O)cc2[o+]c1-c1cc(O)c(O)c(O)c1>>Oc1cc(O)c2c(c1)O[C@H](c1cc(O)c(O)c(O)c1)[C@H](O)C2 ?")

### GPT-5.2 Response

A good EC-number match for that transformation is **anthocyanidin reductase (ANR)**, **EC 1.3.1.77** ✅🧪

Why this fits: your left-hand substrate (the flavylium/polyphenolic cation) looks like an **anthocyanidin** (delphinidin/cyanidin-like), and the product is a **flavan-3-ol** (catechin/gallocatechin-like). **ANR (EC 1.3.1.77)** is exactly the enzyme class that interconverts **anthocyanidins ⇄ flavan-3-ols** via NAD(P)H/NAD(P)+ redox chemistry. ([IUBMB 名称主页][1])

Small practical note: in biochemistry descriptions, the “forward” direction is usually written with **NADPH (or NADH) as the reductant** producing the flavan-3-ol (and NADP+/NAD+ as product), even if a database equation is written in the opposite direction. Experimental work on plant ANRs supports this NADPH-dependent reduction behavior. ([sciencedirect.com][2])

[1]: https://iubmb.qmul.ac.uk/enzyme/EC1/3/1/77.html?utm_source=chatgpt.com "EC 1.3.1.77"
[2]: https://www.sciencedirect.com/science/article/abs/pii/S0003986109003087?utm_source=chatgpt.com "Binding-equilibrium and kinetic studies of anthocyanidin ..."


In [5]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCC(=O)O[C@@H](CO)COC(=O)CCCCCCC/C=C\CCCC.CCCCC/C=C\C/C=C\C/C=C\C/C=C\CCCC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc3c(N)ncnc32)C(O)[C@H]1OP(=O)(O)O>>CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCC(=O)O[C@H](COC(=O)CCC/C=C\C/C=C\C/C=C\C/C=C\CCCCC)COC(=O)CCCCCCC/C=C\CCCC ?")

### GPT-5.2 Response

This looks like an **acyl transfer from a fatty acyl-CoA (CoA thioester reagent) onto a monoacylglycerol**, yielding a **diacylglycerol (DAG)** — i.e., “MAG + acyl-CoA → DAG + CoA” 🧪

Best EC-number recommendation: **Monoacylglycerol O-acyltransferase (MGAT)**, **EC 2.3.1.22** ✅

(Closely related enzymes you sometimes see in the same pathway: **DGAT, EC 2.3.1.20**, but that one typically acylates **DAG → TAG**; your product is DAG, so **MGAT** is the cleaner match.)


In [6]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](OP(=O)(O)O)[C@@H]3O)[C@@H](O)[C@H]2O)C=CC1.OO.[H+]>>NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](OP(=O)(O)O)[C@@H]3O)[C@@H](O)[C@H]2O)c1 ?")

### GPT-5.2 Response

Yes — that transformation is the textbook **NADH → NAD⁺ oxidation with hydrogen peroxide as the electron acceptor**. The best EC-number recommendation is:

**NADH peroxidase, EC 1.11.1.1** ✅

It catalyzes: **NADH + H⁺ + H₂O₂ → NAD⁺ + 2 H₂O**, which matches your reactants (NADH + peroxide + proton) and product (NAD⁺). ([IUBMB 名称主页][1])

If you suspect the enzyme prefers NADPH instead, the close alternative is **NADPH peroxidase, EC 1.11.1.2** (same chemistry but NADPH/NADP⁺). ([IUBMB 名称主页][2])

[1]: https://iubmb.qmul.ac.uk/enzyme/EC1/11/1/1.html?utm_source=chatgpt.com "EC 1.11.1.1"
[2]: https://iubmb.qmul.ac.uk/enzyme/EC1/11/1/?utm_source=chatgpt.com "EC 1.11.1"


In [7]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CCCCCCCC/C=C\CCCCCCCCCC(=O)OC[C@H](COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2ccc(N)nc2=O)C(O)[C@H]1O)OC(=O)CCCCCCCCCC/C=C\CCCCCCCCCCCCC.CCCCCCCCCCCCCCCC(=O)OC[C@H](COP(=O)(O)OC[C@@H](O)CO)OC(=O)CCCCCCCCCCCCCCC>>CC/C=C\CCCCCCCCCC(=O)O[C@H](COC(=O)CCCCCCCCC)COP(=O)(O)OC[C@H](O)COP(=O)(O)OC[C@@H](COC(=O)CCCCCCCCC/C=C\CCCCCCCC)OC(=O)CCCCCCCCCC/C=C\CCCCCCCCCCCCC ?")

### GPT-5.2 Response

This reaction is best matched by **cardiolipin synthase (CMP-forming), EC 2.7.8.41** ✅🧪

Rationale: your reactants correspond to **CDP-diacylglycerol (CDP-DAG)** + **phosphatidylglycerol (PG)**, and the product is **cardiolipin (diphosphatidylglycerol)**; the CMP-forming cardiolipin synthase catalyzes exactly:
**CDP-DAG + PG → cardiolipin + CMP**. ([iubmb.qmul.ac.uk][1])

[1]: https://iubmb.qmul.ac.uk/enzyme/EC2/7/8/41.html?utm_source=chatgpt.com "EC 2.7.8.41"


In [8]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCCCCC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1OP(=O)(O)O.CCCCCCCC/C=C\CCCCCCCC(=O)OC[C@H](CO)OC(=O)CCCCCCCCCCCCCC>>CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCCCCC(=O)OC[C@H](COC(=O)CCCCCCC/C=C\CCCCCCCC)OC(=O)CCCCCCCCCCCCCC ?")

### GPT-5.2 Response

This is an **acyl-CoA–dependent acyl transfer onto a diacylglycerol (DAG)** to form a **triacylglycerol (TAG)** (i.e., **DAG + acyl-CoA → TAG + CoA**) 🧪

Best EC-number recommendation: **Diacylglycerol O-acyltransferase (DGAT)**, **EC 2.3.1.20** ✅


In [9]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: C[S@@+](CC[C@H]([NH3+])C(=O)[O-])C[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1O.O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc([O-])cc(O)c12>>COc1cc(O)c2c(=O)c(O)c(-c3ccc(O)c(O)c3)oc2c1 ?")

### GPT-5.2 Response

This reaction is a classic **SAM-dependent flavonoid O-methylation** (SAM → SAH; phenolic O⁻ → O-Me) ✅🧬

Best EC-number recommendation (given the product looks like a **7-O-methylated flavone/flavonoid**):
• **EC 2.1.1.232 — naringenin 7-O-methyltransferase (a.k.a. flavonoid 7-O-methyltransferase activity)** ([IUBMB 名称主页][1])

If the methoxy were actually on the **B-ring** (common alternatives in flavonoids), then the closest ECs would be:
• **EC 2.1.1.42 — flavone 3′-O-methyltransferase** ([IUBMB 名称主页][2])
• **EC 2.1.1.75 — apigenin 4′-O-methyltransferase** ([IUBMB 名称主页][3])

And for a related “7-O-methylation on a flavonol scaffold” example EC:
• **EC 2.1.1.82 — 3-methylquercetin 7-O-methyltransferase** ([Enzyme][4])

If you want, paste the **compound name** (e.g., luteolin → ?), or tell me **which hydroxyl is methylated (7-O vs 3′-O vs 4′-O)**, and I’ll pin it to the single most specific EC without hedging.

[1]: https://iubmb.qmul.ac.uk/enzyme/EC2/1/1/232.html?utm_source=chatgpt.com "EC 2.1.1.232"
[2]: https://iubmb.qmul.ac.uk/enzyme/EC2/1/1/42.html?utm_source=chatgpt.com "EC 2.1.1.42"
[3]: https://iubmb.qmul.ac.uk/enzyme/EC2/1/1/75.html?utm_source=chatgpt.com "EC 2.1.1.75"
[4]: https://enzyme.expasy.org/EC/2.1.1.82?utm_source=chatgpt.com "ENZYME - 2.1.1.82 3-methylquercetin 7-O-methyltransferase"


In [10]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CO[C@@H]1[C@H](O)[C@@H](COP(=O)([O-])OP(=O)([O-])OC[C@H]2O[C@@H]([n+]3cccc(C(N)=O)c3)[C@H](O)[C@@H]2O)O[C@H]1n1cnc2c(N)ncnc21.CC1=C(O)C=C[C@]2(C)CC[C@H]3[C@H](C)C(=O)O[C@@H]3C12>>CC1=C2C3OC(=O)C(C)C3CCC2(C)C=CC1=O ?")

### GPT-5.2 Response

This looks like an **NAD⁺-dependent oxidation of a steroid alcohol to the corresponding ketone** (a typical “3β-hydroxy steroid → 3-oxo steroid” step) 🧪

The best EC-number match to recommend is:

**EC 1.1.1.145 — 3β-hydroxy-Δ⁵-steroid dehydrogenase (NAD⁺)**. ([IUBMB 名称主页][1])

(If your intended biology also includes the accompanying **Δ⁵→Δ⁴ double-bond isomerization** that often pairs with this oxidation in steroid pathways, that’s commonly treated as the same bifunctional system in practice, but the core oxidation step is still captured by **EC 1.1.1.145**.)

[1]: https://iubmb.qmul.ac.uk/enzyme/EC1/1/1/145.html?utm_source=chatgpt.com "EC 1.1.1.145"


In [11]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CC/C=C\C/C=C\C/C=C\C/C=C\CCCCCCC(=O)O[C@@H](CO)COC(=O)CCCCCCCCC/C=C\CCCCCCCC.CCCCC/C=C\C/C=C\C/C=C\CCCCC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1OP(=O)(O)O>>CC/C=C\C/C=C\C/C=C\C/C=C\CCCCCCC(=O)O[C@H](COC(=O)CCCC/C=C\C/C=C\C/C=C\CCCCC)COC(=O)CCCCCCCCC/C=C\CCCCCCCC ?")

### GPT-5.2 Response

This is an **acyl-CoA–dependent acylation of a monoacylglycerol (MAG) to form a diacylglycerol (DAG)**, i.e. **MAG + acyl-CoA → DAG + CoA** 🧪

Recommended enzyme (EC): **Monoacylglycerol O-acyltransferase (MGAT), EC 2.3.1.22** ✅


In [12]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CCCCCCCC/C=C\CCCCCCCC(=O)OC[C@H](CO)OC(=O)CCCCCCCCCCCCCCCCCCCCCCC.CCCCCCCC/C=C\CCCCCCCCCCCC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1OP(=O)(O)O>>CCCCCCCC/C=C\CCCCCCCCCCCC(=O)OC[C@H](COC(=O)CCCCCCC/C=C\CCCCCCCC)OC(=O)CCCCCCCCCCCCCCCCCCCCCCC ?")

### GPT-5.2 Response

This reaction is **acyl-CoA–dependent acylation of a diacylglycerol (DAG) to a triacylglycerol (TAG)** — i.e., **DAG + acyl-CoA → TAG + CoA** 🧪

Recommended enzyme (EC): **Diacylglycerol O-acyltransferase (DGAT), EC 2.3.1.20** ✅


In [13]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-]>>O=C(O)/C=C(/CC(=O)O)C(=O)O ?")

### GPT-5.2 Response

That reaction is **citrate ⇄ cis-aconitate + H₂O** (a dehydration step). The standard enzyme that catalyzes it is **aconitate hydratase (aconitase)**, **EC 4.2.1.3** ✅🧪. ([Enzyme][1])

[1]: https://enzyme.expasy.org/EC/4.2.1.3?utm_source=chatgpt.com "ENZYME - 4.2.1.3 aconitate hydratase"


In [14]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CC(=O)c1ccccc1.NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](OP(=O)(O)O)[C@@H]3O)[C@@H](O)[C@H]2O)C=CC1.[H+]>>CC(O)c1ccccc1 ?")

### GPT-5.2 Response

That’s a straightforward **NADH-dependent reduction of acetophenone → 1-phenylethanol** (a “ketoreductase” / alcohol dehydrogenase-type step) 🧪

Best EC-number recommendation:
• **Alcohol dehydrogenase (NAD⁺), EC 1.1.1.1** ✅
(physiologically it’s written as alcohol → ketone with NAD⁺, but the reverse direction is exactly your ketone → alcohol with NADH)

If you care about stereochemistry of the alcohol product (R vs S), you can refine to stereospecific ADHs:
• **(R)-specific alcohol dehydrogenase, EC 1.1.1.2**
• **(S)-specific alcohol dehydrogenase, EC 1.1.1.239**


In [15]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)([O-])OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](O)[C@@H]3O)[C@@H](O)[C@H]2O)C=CC1.O=c1c(=O)c2cccc3ccc4cccc1c4c32.[H+]>>Oc1c(O)c2cccc3ccc4cccc1c4c32 ?")

### GPT-5.2 Response

This reaction is a classic **NADH-dependent quinone reduction** (quinone → hydroquinone). ✅🧪

Recommended EC number: **EC 1.6.5.2 — NAD(P)H dehydrogenase (quinone)**, which catalyzes **NAD(P)H + H⁺ + a quinone ⇄ NAD(P)⁺ + a hydroquinone**. ([IUBMB 名称主页][1])

[1]: https://iubmb.qmul.ac.uk/enzyme/EC1/6/5/2.html?utm_source=chatgpt.com "EC 1.6.5.2"


In [16]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CCCCC/C=C\CCCCCCCC(=O)OC[C@H](COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2ccc(N)nc2=O)C(O)[C@H]1O)OC(=O)CCCCCCCCCCC/C=C\CCCCCC.CCCCCCCCCCCCCCCC(=O)OC[C@H](COP(=O)(O)OC[C@@H](O)CO)OC(=O)CCCCCCCCCCCCCCC>>CC/C=C\CCCCCCCCCC(=O)OC[C@H](COP(=O)(O)OC[C@H](O)COP(=O)(O)OC[C@@H](COC(=O)CCCCCCC/C=C\CCCCC)OC(=O)CCCCCCCCCCC/C=C\CCCCCC)OC(=O)CCCCCCCCCCCCCC ?")

### GPT-5.2 Response

This is the hallmark **cardiolipin (diphosphatidylglycerol) formation** reaction: **CDP-diacylglycerol (CDP-DAG) + phosphatidylglycerol (PG) → cardiolipin + CMP** 🧪

Recommended enzyme (EC): **Cardiolipin synthase (CMP-forming), EC 2.7.8.41** ✅


In [17]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CCCC.NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](O)[C@@H]3O)[C@@H](O)[C@H]2O)C=CC1.O=O.[H+]>>CCC(C)O ?")

### GPT-5.2 Response

A good EC-number match is **butane monooxygenase (soluble), EC 1.14.13.230** ✅🧪

It catalyzes **butane + NADH + H⁺ + O₂ → butanol + NAD⁺ + H₂O**, which matches your use of an NTP cofactor (NADH), O₂, and formation of a butanol product (your SMILES specifies 2-butanol, i.e., subterminal hydroxylation). ([IUBMB 名称主页][1])

[1]: https://iubmb.qmul.ac.uk/enzyme/EC1/14/13/230.html?utm_source=chatgpt.com "EC 1.14.13.230"


In [18]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCCCCC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1OP(=O)(O)O.CCCCC/C=C\C/C=C\C/C=C\C/C=C\CCCCCC(=O)O[C@@H](CO)COC(=O)CCCCCCCCCCCCCCCCCCC>>CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCCCCC(=O)OC[C@H](COC(=O)CCCCCCCCCCCCCCCCCCC)OC(=O)CCCCC/C=C\C/C=C\C/C=C\C/C=C\CCCCC ?")

### GPT-5.2 Response

This is an **acyl-CoA–dependent acyl transfer onto a diacylglycerol (DAG)** to form a **triacylglycerol (TAG)** (i.e., **DAG + acyl-CoA → TAG + CoA**) 🧪

Best EC-number recommendation: **Diacylglycerol O-acyltransferase (DGAT)**, **EC 2.3.1.20** ✅


In [19]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CCCCCCCCCCC/C=C\CCCCCCCCCC(=O)O[C@H](COC(=O)CCCCCCCCCCCCCCCCCCC)COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2ccc(N)nc2=O)C(O)[C@H]1O.CCCCCCCCCCCCCCCC(=O)OC[C@H](COP(=O)(O)OC[C@@H](O)CO)OC(=O)CCCCCCCCCCCCCCC>>CCCCC/C=C\CCCCCCCC(=O)O[C@H](COC(=O)CCCCCCCCCCCCC)COP(=O)(O)OC[C@H](O)COP(=O)(O)OC[C@@H](COC(=O)CCCCCCCCCCCCCCCCCCC)OC(=O)CCCCCCCCC/C=C\CCCCCCCCCCC ?")

### GPT-5.2 Response

This reaction is best matched by **cardiolipin synthase (CMP-forming), EC 2.7.8.41** ✅🧪

Reason: your left side is consistent with **CDP-diacylglycerol (CDP-DAG)** reacting with **phosphatidylglycerol (PG)** to form **cardiolipin (diphosphatidylglycerol)** (and CMP is typically released, even if not explicitly written in the SMILES). That’s exactly the defining chemistry for EC 2.7.8.41. ([iubmb.qmul.ac.uk][1])

[1]: https://iubmb.qmul.ac.uk/enzyme/EC2/7/8/41.html?utm_source=chatgpt.com "EC 2.7.8.41"


In [20]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CS.Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])OP(=O)([O-])OP(=O)([O-])[O-])[C@@H](O)[C@H]1O.O=C([O-])CCCCCCCCCCCCCCCCc1ccc(O)cc1>>CSC(=O)CCCCCCCCCCCCCCCCc1ccc(O)cc1 ?")

### GPT-5.2 Response

A sensible EC-level match is an **acid–thiol ligase (thioester-forming)**, i.e. an **acyl-thiol ligase** in the **EC 6.2.1.-** class ✅🧪

Why: your reaction is essentially “carboxylate + thiol (methanethiol) + NTP → thioester (methyl thioester)”, which is the defining chemistry of **EC 6.2.1 (acid–thiol ligases)**. In most annotated pathways the energy donor is written as **ATP**, but some systems can be annotated/represented with another NTP (your substrate string looks like **GTP**).

If you want one concrete “closest named” EC to try first (family-wise): **long-chain acyl-CoA synthetase / long-chain fatty acid–CoA ligase, EC 6.2.1.3** is the most common scaffold that does carboxylate activation + thioester formation (even though its physiological thiol is CoA rather than methanethiol).


In [21]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CCCCCCCCCCCCCCCC(=O)OC[C@@H](O)COP(=O)([O-])OC[C@@H](O)CO.CCCCCCCCCCCCCCCCCC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)([O-])OP(=O)([O-])OC[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1OP(=O)([O-])[O-]>>CCCCCCCCCCCCCCCCCC(=O)O[C@H](COC(=O)CCCCCCCCCCCCCCC)COP(=O)([O-])OC[C@@H](O)CO ?")

### GPT-5.2 Response

This reaction is the **acyl-CoA–dependent acylation of lysophosphatidylglycerol (LPG) to phosphatidylglycerol (PG)** — i.e., “LPG + acyl-CoA → PG + CoA” 🧪

The best EC recommendation is therefore:

**Acyl-CoA:1-acyl-sn-glycero-3-phosphoglycerol O-acyltransferase (lysophosphatidylglycerol acyltransferase, LPGAT)**
**EC 2.3.1.-** ✅ (assigned at the sub-subclass level, but commonly not given a specific terminal number)

KEGG explicitly defines this exact reaction as **R09036** and links it to **LPGAT1 (K13514) with EC:2.3.1.-**. ([genome.jp][1])

[1]: https://www.genome.jp/dbget-bin/www_bget?rn%3AR09036=&utm_source=chatgpt.com "KEGG REACTION: R09036"


In [22]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: O=c1c([O-])c(-c2ccc(O)cc2)oc2cc(O)cc(O)c12.O=c1nc(O)ccn1[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O)O[C@@H]2O[C@H](CO)[C@H](O)[C@H](O)[C@H]2O)[C@@H](O)[C@H]1O>>O=c1c(O[C@@H]2O[C@H](CO)[C@H](O)[C@H](O)[C@H]2O)c(-c2ccc(O)cc2)oc2cc([O-])cc(O)c12 ?")

### GPT-5.2 Response

A strong EC-number match here is **flavonol 3-O-glucosyltransferase**, **EC 2.4.1.91** ✅🍬

Reason: your reaction shows transfer of a **glucosyl group from UDP-glucose** onto a **flavonol (phenolic O-glucosylation)** to form a **flavonol O-glucoside**, which is exactly the defining chemistry of EC 2.4.1.91. ([Enzyme][1])

(If the acceptor were an **anthocyanidin** rather than a flavonol, the closest alternative would be **anthocyanidin 3-O-glucosyltransferase, EC 2.4.1.115**, but that’s generally distinguished from the flavonol enzyme. ([Enzyme][2]))

[1]: https://enzyme.expasy.org/EC/2.4.1.91?utm_source=chatgpt.com "ENZYME - 2.4.1.91 flavonol 3-O-glucosyltransferase"
[2]: https://enzyme.expasy.org/EC/2.4.1.115?utm_source=chatgpt.com "ENZYME - 2.4.1.115 anthocyanidin 3-O-glucosyltransferase"


In [23]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CC(C)=CCC/C(C)=C/COP(=O)([O-])OP(=O)([O-])[O-]>>CC1=CCC2CC1C2(C)C ?")

### GPT-5.2 Response

This reaction is the classic cyclization of **geranyl diphosphate (GPP)** to **α-pinene + diphosphate** 🌲🧪

Most appropriate EC recommendation:
• **α-Pinene synthase (geranyl-diphosphate diphosphate-lyase, cyclizing)**
 – **EC 4.2.3.121** for **(+)-α-pinene synthase** ([IUBMB 名称主页][1])
 – **EC 4.2.3.119** for **(–)-α-pinene synthase** ([IUBMB 名称主页][2])

Because your product SMILES doesn’t specify stereochemistry, either enantiomer-specific enzyme could fit.

[1]: https://iubmb.qmul.ac.uk/enzyme/EC4/2/3/121.html?utm_source=chatgpt.com "EC 4.2.3.121"
[2]: https://iubmb.qmul.ac.uk/enzyme/EC4/2/3/119.html?utm_source=chatgpt.com "EC 4.2.3.119"


In [24]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CCCCCCCCCCC[C@@H](O)CC(=O)SC.CC(=O)N[C@H]1[C@@H](OP(=O)([O-])OP(=O)([O-])OC[C@H]2O[C@@H](n3ccc(=O)[nH]c3=O)[C@H](O)[C@@H]2O)O[C@H](CO)[C@@H](O)[C@@H]1O>>CCCCCCCCCCC[C@@H](O)CC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H](OP(=O)([O-])OP(=O)([O-])OC[C@H]2O[C@@H](n3ccc(=O)[nH]c3=O)[C@H](O)[C@@H]2O)[C@@H]1NC(C)=O ?")

### GPT-5.2 Response

Yes ✅ — the best-matching EC number for that chemistry is:

**acyl-[acyl-carrier-protein]—UDP-N-acetylglucosamine O-acyltransferase (LpxA), EC 2.3.1.129**. ([IUBMB 名称主页][1])

Why this fits (briefly): your reaction is an **O-acyl transfer onto UDP-N-acetylglucosamine (UDP-GlcNAc)** using a **(3R)-3-hydroxyacyl thioester donor** (your donor is a small thioester analog, not ACP-bound), yielding **UDP-3-O-acyl-GlcNAc** — exactly the defining transformation for **LpxA / EC 2.3.1.129** in lipid A biosynthesis. ([IUBMB 名称主页][1])

[1]: https://iubmb.qmul.ac.uk/enzyme/EC2/3/1/129.html?utm_source=chatgpt.com "EC 2.3.1.129"


In [25]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCC(=O)O[C@@H](CO)COC(=O)CCCCCCCCC/C=C\CCCCCC.CCCCC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1OP(=O)(O)O>>CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCC(=O)O[C@H](COC(=O)CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCCCC)COC(=O)CCCCCCCCC/C=C\CCCCCC ?")

### GPT-5.2 Response

This reaction is **acyl-CoA–dependent acylation of a diacylglycerol (DAG) to a triacylglycerol (TAG)** — i.e., **DAG + acyl-CoA → TAG + CoA** 🧪

Recommended enzyme (EC):
**Diacylglycerol O-acyltransferase (DGAT), EC 2.3.1.20** ✅


In [26]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCCC(=O)O[C@@H](CO)COC(=O)CCCCCCCCCCCCCCCCCCCCCCC.CCCCCCCC/C=C\CCCCCCCC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1OP(=O)(O)O>>CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCCC(=O)O[C@H](COC(=O)CCCCCCC/C=C\CCCCCCCC)COC(=O)CCCCCCCCCCCCCCCCCCCCCCC ?")

### GPT-5.2 Response

This is another classic **acyl-CoA–dependent acylation of a diacylglycerol (DAG) to a triacylglycerol (TAG)**:
**DAG + acyl-CoA → TAG + CoA** 🧪

Recommended enzyme (EC): **Diacylglycerol O-acyltransferase (DGAT), EC 2.3.1.20** ✅


In [27]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CCCCCCCC/C=C\CCCCCCCCCCCC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1OP(=O)(O)O.CCCCCCCCCCCCCCC(=O)OC[C@H](CO)OC(=O)CCCCCCCCCCCCC>>CCCCCCCC/C=C\CCCCCCCCCCCC(=O)OC[C@H](COC(=O)CCCCCCCCCCCCCC)OC(=O)CCCCCCCCCCCCC ?")

### GPT-5.2 Response

This is **acyl-CoA–dependent acylation of a diacylglycerol (DAG) to a triacylglycerol (TAG)**:
**DAG + acyl-CoA → TAG + CoA** 🧪

Recommended enzyme (EC): **Diacylglycerol O-acyltransferase (DGAT)**, **EC 2.3.1.20** ✅


In [28]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CCCCC/C=C/C/C=C/CCCCCCCC(=O)O.O=O>>CCCCC[C@H](/C=C\C=C\CCCCCCCC(=O)[O-])OO ?")

### GPT-5.2 Response

This is an **O₂ addition (dioxygenation) to a polyunsaturated fatty acid** to form a **fatty-acid hydroperoxide** (a chiral –OOH product with a conjugated diene) 🧪🫧

The most appropriate EC recommendation is a **lipoxygenase**:

• **Linoleate 13S-lipoxygenase, EC 1.13.11.12** (a canonical “linoleate + O₂ → hydroperoxyoctadecadienoate” reaction). ([IUBMB 名称主页][1])

Depending on the exact regio-/stereochemistry (which your SMILES doesn’t fully pin down), a close alternative is:

• **Linoleate 9S-lipoxygenase, EC 1.13.11.58** (same substrate class, different oxygenation position). ([Enzyme][2])

If you tell me whether the product is the **9-HPOD vs 13-HPOD** (or give a named lipid like 9S-HpODE / 13S-HpODE), I’ll pick the single best EC unambiguously.

[1]: https://iubmb.qmul.ac.uk/enzyme/EC1/13/11/12.html?utm_source=chatgpt.com "EC 1.13.11.12"
[2]: https://enzyme.expasy.org/EC/1.13.11.12?utm_source=chatgpt.com "ENZYME - 1.13.11.12 linoleate 13S-lipoxygenase"


In [29]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: N#CCCO.O>>O=C(O)CCO ?")

### GPT-5.2 Response

Yes — the cleanest single-enzyme EC match is:

**Nitrilase, EC 3.5.5.1** ✅

Your reaction is essentially hydration of a nitrile (3-hydroxypropionitrile–like) to the corresponding carboxylic acid (3-hydroxypropionic acid). **Nitrilase (EC 3.5.5.1)** catalyzes **direct conversion of nitriles to carboxylic acids (typically releasing NH₃)**, i.e. it’s the one-step route. ([Genome.jp][1])

If you’re okay with a two-step enzymatic route (often seen industrially/biotech), the alternative is:

* **Nitrile hydratase, EC 4.2.1.84** (nitrile → amide), then
* **Amidase, EC 3.5.1.4** (amide → acid + NH₃). ([PMC][2])

[1]: https://www.genome.jp/dbget-bin/www_bget?ec%3A3.5.5.1=&utm_source=chatgpt.com "KEGG ENZYME: 3.5.5.1"
[2]: https://pmc.ncbi.nlm.nih.gov/articles/PMC7193024/?utm_source=chatgpt.com "Recent Advances and Promises in Nitrile Hydratase"


In [30]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCCCCC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1OP(=O)(O)O.CC/C=C\C/C=C\C/C=C\C/C=C\CCCCCCC(=O)O[C@@H](CO)COC(=O)CCC/C=C\C/C=C\C/C=C\CCCCCCCC>>CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCCCCC(=O)OC[C@H](COC(=O)CCC/C=C\C/C=C\C/C=C\CCCCCCCC)OC(=O)CCCCCC/C=C\C/C=C\C/C=C\C/C=C\CC ?")

### GPT-5.2 Response

Yes — that reaction is an **acyl-CoA–dependent acylation of a diacylglycerol (DAG) to form a triacylglycerol (TAG)** (i.e., **DAG + acyl-CoA → TAG + CoA**) 🧪

Recommended EC number: **Diacylglycerol O-acyltransferase (DGAT), EC 2.3.1.20** ✅


In [31]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: Cc1cc2c(cc1C)N(C)c1[nH]c(=O)[nH]c(=O)c1N2.COc1ccc(C[NH2+]CCc2ccc(O)cc2)cc1O.O=O>>COc1ccc2c3c1O[C@H]1CC(=O)C=C[C@@]31CCNC2 ?")

### GPT-5.2 Response

A good EC-number match is **reticuline oxidase (berberine bridge enzyme; “BBE”) — EC 1.21.3.3** ✅🧪 ([Enzyme][1])

Why it fits: this enzyme class is a **flavin-dependent oxidase** that uses **O₂** to drive an **oxidative intramolecular C–C bond–forming cyclization** in benzylisoquinoline alkaloid scaffolds (the “berberine bridge” chemistry), typically producing **H₂O₂** as a coproduct in the physiological direction. ([Enzyme][1])

[1]: https://enzyme.expasy.org/EC/1.21.3.3?utm_source=chatgpt.com "ENZYME - 1.21.3.3 reticuline oxidase"


In [32]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat(r"Can you recommend an enzyme (EC number) that may catalyze this reaction: CN[C@@H](COP(=O)([O-])OCC(C)(C)[C@@H](O)C(=O)NCCC(=O)NCCS)C(C)=O.C/C(=C\C(=O)OCCCCCCCCC(=O)[O-])C[C@@H]1OC[C@](O)(CC2OC2[C@@H](C)[C@H](C)O)[C@@H](O)[C@H]1O.Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])OP(=O)([O-])OP(=O)([O-])[O-])[C@@H](O)[C@H]1O>>CN[C@@H](COP(=O)([O-])OCC(C)(C)[C@@H](O)C(=O)NCCC(=O)NCCSC(=O)CCCCCCCCOC(=O)/C=C(\C)C[C@@H]1OC[C@](O)(CC2OC2[C@@H](C)[C@H](C)O)[C@@H](O)[C@H]1O)C(C)=O ?")

### GPT-5.2 Response

Yes — the chemistry here is an **acid → thioester “activation + transfer”** onto a **4′-phosphopantetheine thiol (ACP/PCP-like carrier)**, driven by an NTP. That’s the hallmark of an **acid–thiol ligase**.

Best EC-number recommendation (most specific fit):
• **Long-chain-fatty-acid:[acyl-carrier protein] ligase (AMP-forming)**, **EC 6.2.1.20** ✅
This class ligates a carboxylate onto an **acyl-carrier protein (ACP)** to form an **acyl-ACP thioester**. ([IUBMB 名称主页][1])

Practical alternative (if the biological thiol were CoA rather than ACP/PCP):
• **Long-chain-fatty-acid—CoA ligase (acyl-CoA synthetase)**, **EC 6.2.1.3** ([IUBMB 名称主页][2])

[1]: https://iubmb.qmul.ac.uk/enzyme/EC6/2/1/20.html?utm_source=chatgpt.com "EC 6.2.1.20"
[2]: https://iubmb.qmul.ac.uk/enzyme/EC6/2/1/3.html?utm_source=chatgpt.com "EC 6.2.1.3"
